# `notebooks/MARS.ipynb` — What it does

This document explains what the notebook `notebooks/MARS.ipynb` does end-to-end: what it loads, which pipeline functions it calls (Systems 1–3), how the System 2↔3 loop works, and what artifacts it writes to disk.

## High-level purpose

The notebook is an **orchestrator** for the repo’s “hierarchical system”:

- **System 1 (runs once)**: extract required material properties \(W\) and hard constraints \(U\) from a user query, using RAG + the PFAS KG during question answering.
- **System 2 (runs in a loop)**: propose and validate candidate substitute materials, using RAG and a **material-informed subgraph constructed from both KGs**.
- **System 3 (runs in a loop)**: manufacturability assessment of the System 2 candidate; if blocked, feed constraints back into System 2 and repeat.

Conceptually:

```mermaid
flowchart TD
  system1[System1_propertyExtraction] --> system2[System2_materialDiscovery]
  system2 --> system3[System3_manufacturability]
  system3 -->|blocked_feedback| system2
  system3 -->|manufacturable| endNode[Stop]
```

## Inputs

The notebook’s primary “input” is a **research query** provided inline as Python variables:

- `sentence`: free-text problem statement / prompt
- `material_X`: the material being replaced
- `application_Y`: a compact application context string
- `keywords`: typically `[material_X, application_Y]`

All other behavior (paths, model names, KG filenames, etc.) is driven by `config/config.yaml`.

## Setup phase (imports + config)

The notebook:

1. Locates the repo root (`project_root`) and inserts it into `sys.path`.
2. Imports utilities, agents, and pipeline entrypoints from `src/`.
3. Loads configuration via `src.config.load_config()` from `config/config.yaml`.

Key imported pipeline functions:

- `run_fixed_pipeline` (System 1) from `src/pipelines/material_requirements.py`
- `run_material_discovery_pipeline` (System 2) from `src/pipelines/material_discovery.py`
- `run_manufacturability_assessment_pipeline` (System 3) from `src/pipelines/manufacturability_assessment.py`

## Initialize LLM + embeddings

The notebook creates:

- an LLM wrapper (`llm_instance = llm(llm_config)`) and a callable `generate = llm_instance.generate_cli`
- a sentence-transformer embedding model (`SentenceTransformer(config["embeddings"]["model_name"])`)
- a `TransformerEmbeddingFunction` for ChromaDB collections

These objects are then passed into agents (manager/scientists) and RAG components.

## Load knowledge graphs + node embeddings

The notebook loads three KGs from `config["data"]["graphs"]`:

- **Material Properties KG** (`G_materialproperties`) + embeddings (`node_embeddings_materialproperties`)
- **PFAS KG** (`G_pfas`) + embeddings (`node_embeddings_pfas`)
- **Patents KG** (`G_patents`) + embeddings (`node_embeddings_patents`)

Implementation notes:

- Graphs are loaded from GraphML (`nx.read_graphml(...)`).
- It copies edge attribute `title` into `relation` and computes PageRank into node attribute `pr`.
- Embeddings are loaded from the configured `.pkl` files using `GraphReasoning.load_embeddings`.

## Load ChromaDB corpora (RAG sources)

Using `chromadb.PersistentClient`, it loads several ChromaDB databases/collections:

- **PFAS** corpus (used in System 1)
- **Patents** corpus (used in System 2; also used as a System 3 evidence source)
- **MaterialDB** corpus (used in System 2; also used as a System 3 evidence source)
- Optional: manufacturing textbooks and spec sheets (used in System 3) if present on disk

For System 3, it builds a `MultiAnalyst(process_analysts)` which aggregates multiple RAG sources.

## Initialize per-system agents and helpers

### System 1 agents

System 1 uses:

- `ResearchAnalyst` over the PFAS ChromaDB
- `ResearchManager`
- `ResearchAssistant`
- `ResearchScientist` (material properties KG) — **note**: System 1 no longer generates graphs in the canonical pipeline; this object may still be constructed but should not be relied on for graph outputs
- `pfas_scientist_s1` (`ResearchScientist` over PFAS KG) used during question answering (System 1’s per-question KG lookup)

### System 2 agents and helpers

System 2 uses:

- `ResearchScientist` initialized with **both** KGs (material properties + patents)
- Two `ResearchAnalyst`s (Patents + MaterialDB) wrapped as a `MultiAnalyst`
- `RejectedCandidateTracker` shared across iterations
- `MaterialDatabase` loaded from JSON
- `SubgraphProcessor` (relevance filtering utilities; the canonical pipeline no longer consumes System 1 `subgraph_data`)
- `MaterialGrounding` for both KGs:
  - `material_grounding_material` (material properties KG)
  - `material_grounding_patents` (patents KG)
- `Step1Cache` (in-memory cache; notebook also keeps an explicit `cached_substitution_result`)

### System 3 agents

System 3 uses:

- `process_analyst` (a `MultiAnalyst` over manufacturing-related sources)
- its own manager/analyst instances (created per iteration)

## Run metadata / logging layout

The notebook writes a run record and per-system outputs under:

- `pipeline_logs/`

and uses `ChatLogger` to write structured LLM interaction logs under the logs directory’s `chats/` subfolder (exact location depends on `ChatLogger` implementation + config).

The notebook also constructs a `pipeline_run` dict that stores:

- start/end timestamps and durations
- result file paths
- System 2↔3 loop iteration metadata

## System 1: property extraction (runs once)

System 1 is executed by calling:

- `system1_result = run_fixed_pipeline(...)`

System 1 produces (in-memory):

- extracted property keywords: `system1_result["extracted_keywords"]`
- extracted hard constraints: `system1_result["extracted_constraints"]`
- additional QA artifacts (questions, answers, RAG docs, etc.)

The notebook then constructs:

- `properties_W = {"required": extracted_keywords, "target_values": {}}`
- `constraints_U = extracted_constraints`

### System 1 artifacts written

The notebook writes `system1_<run_id>.json` containing (at least):

- `run_id`, `timestamp`, `sentence`, `keywords`
- `material_X`, `application_Y`
- `properties_W`, `extracted_keywords`
- `extracted_constraints`
- `chat_log_path`

Important: **System 1 no longer serializes or emits `subgraph`/`keyword_connections`** in the canonical pipeline.

## System 2 ↔ System 3 loop

After System 1, the notebook enters a while-loop for up to `max_iterations`.

At each iteration:

1. It creates a new `system2_run_id` and `ChatLogger`.
2. It constructs System 2 agents (manager, analysts, material scientist).
3. It calls:

   - `system2_result = run_material_discovery_pipeline(...)`

4. If System 2 succeeds and proposes a candidate, it runs:

   - `system3_result = run_manufacturability_assessment_pipeline(system2_result, ...)`

5. If System 3 returns `status == "blocked"`, it extracts feedback constraints and loops back.
6. If System 3 returns `status == "manufacturable"`, it stops.

### “Compute once” Step 1 behavior (material-informed subgraph)

The notebook keeps:

- `cached_substitution_result`

and passes it into `run_material_discovery_pipeline(..., substitution_result=cached_substitution_result)` so that System 2’s Step 1 (graph/subgraph construction + mapping) runs only once and is reused across System 2↔3 loop iterations.

Within the current canonical implementation of System 2 Step 1:

- it constructs two intermediate per-KG subgraphs in-memory (`subgraph_matkg` and `subgraph_patkg`)
- merges them (unify-by-title via embeddings) into a single `material_informed_subgraph`
- **persists only** the merged `material_informed_subgraph` as a subgraph artifact under `pipeline_logs/subgraphs/`

## Outputs written by System 2 and System 3

Across iterations, the notebook writes:

- `system2_<run_id>.json` (System 2 results)
- `system3_<run_id>.json` (System 3 results) when System 2 succeeds enough to evaluate manufacturability
- a `pipeline_run_<base_run_id>.json` record linking all stage outputs and loop iterations

Additionally, the repo includes helper notebooks for visualization that expect these log layouts.

## Common reasons the notebook can fail

- **Missing data paths**: `config/config.yaml` paths point to ORCD locations by default.
- **ChromaDB not present**: patent/materialdb databases must exist at configured locations.
- **Graph/embedding mismatch**: embedding keys must match node IDs in the loaded GraphML.
- **LLM endpoint**: `llm.base_url` must be reachable and compatible with the wrapper.

